> **Setup notebook — not part of `scripts/run_phase1.py`.**
> Runs once to produce `backend/data/raw/visayas_power_raw.geojson`,
> the OSM dump that the Phase 1 pipeline (notebook 02 onwards) reads.
> The extraction cell is idempotent and skipped if the file exists;
> set `FORCE_REFRESH = True` to re-pull from Overpass.

# Phase 1A.1 — OSM Extraction

Extract Visayas power infrastructure from OpenStreetMap and stage the raw dump for the coverage audit.

**Tags pulled:** `power=line | cable | substation | tower | generator`

**bbox:** `(123.0, 9.0, 126.5, 13.0)` = (west_lon, south_lat, east_lon, north_lat)

Output: `backend/data/raw/visayas_power_raw.geojson`

**Note:** This hits the Overpass API and takes 1–3 minutes. The cell is idempotent — re-running it overwrites the file. Skip the extraction cell if the file already exists and is current.

In [1]:
from pathlib import Path
import osmnx as ox
import geopandas as gpd

RAW = Path('../backend/data/raw/visayas_power_raw.geojson')
RAW.parent.mkdir(parents=True, exist_ok=True)
print('Target file:', RAW.resolve())
print('Exists:', RAW.exists(), '(size:', RAW.stat().st_size if RAW.exists() else 'n/a', ')')

Target file: /Users/julius/polymath/Projects/powergrid/backend/data/raw/visayas_power_raw.geojson
Exists: True (size: 29205648 )


In [2]:
# Extraction cell. Re-run only when you want to refresh from OSM.
FORCE_REFRESH = False

if FORCE_REFRESH or not RAW.exists():
    tags = {'power': ['line', 'cable', 'substation', 'tower', 'generator']}
    # bbox = (left, bottom, right, top) = (west_lon, south_lat, east_lon, north_lat)
    visayas_bbox = (123.0, 9.0, 126.5, 13.0)
    print('Extracting Visayas power infrastructure from OSM...')
    gdf = ox.features_from_bbox(bbox=visayas_bbox, tags=tags)
    print(f'Total features: {len(gdf)}')
    gdf.to_file(RAW, driver='GeoJSON')
    print(f'Saved → {RAW}')
else:
    print(f'Using cached {RAW}. Set FORCE_REFRESH=True to re-extract.')

Using cached ../backend/data/raw/visayas_power_raw.geojson. Set FORCE_REFRESH=True to re-extract.


## Quick inventory

Load the raw file and print the high-level counts that decide whether Phase 1C is minor or major.

In [3]:
gdf = gpd.read_file(RAW)
print(f'Total features: {len(gdf)}')
print(f'Geometry types: {gdf.geom_type.value_counts().to_dict()}')
print()
print('Power tag distribution:')
print(gdf['power'].value_counts())

Total features: 19607
Geometry types: {'Point': 18864, 'LineString': 631, 'Polygon': 112}

Power tag distribution:
power
tower         18853
line            589
substation      121
cable            44
Name: count, dtype: int64


/Users/julius/polymath/Projects/powergrid/.venv/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: Several features with id = 1237981974 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


In [4]:
if 'voltage' in gdf.columns:
    print('Top 20 voltage tag values (raw strings):')
    print(gdf['voltage'].value_counts().head(20))
    print()
    print(f'Rows missing voltage tag: {gdf["voltage"].isna().sum()} / {len(gdf)}')

Top 20 voltage tag values (raw strings):
voltage
138000                                                                       186
230000                                                                       100
69000                                                                         73
350000                                                                        26
138000;69000                                                                  14
60000                                                                         11
3000                                                                          10
30000                                                                          9
69000;13200                                                                    8
69000;23000                                                                    7
230000;138000                                                                  3
350000;230000;138000                                        